<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

# Data Quality & Profiling Report

## Objective
Generate comprehensive data quality report with validation checks using ydata-profiling

## Validation Checks
1. Missing values and completeness
2. Duplicate entries
3. Schema and data type validation
4. Range and format checks (rating scale 1-5)
5. Statistical analysis and distributions

## Output
- HTML profiling report
- JSON data quality summary
- Data validation report

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ydata-profiling
from ydata_profiling import ProfileReport

# Validation
import json
from datetime import datetime
import os

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Step 2: Load Raw Data

In [2]:
print("Loading raw data...\n")

# Load raw data
#df_raw = pd.read_csv(r'e:/dm4ml project/data/processed/final_dataset.csv')
df_raw = pd.read_csv("data/processed/final_dataset.csv")

print(f"✓ Raw data loaded")
print(f"  Shape: {df_raw.shape}")
print(f"  Columns: {df_raw.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df_raw.head())

Loading raw data...



✓ Raw data loaded
  Shape: (600000, 27)
  Columns: ['user_id', 'product_id', 'interaction_type', 'user_rating', 'timestamp', 'run_id', 'title', 'description', 'category', 'price', 'discountPercentage', 'product_rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'dimensions', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'reviews', 'returnPolicy', 'minimumOrderQuantity', 'meta', 'images', 'thumbnail']

First 5 rows:
   user_id  product_id interaction_type  user_rating  \
0      275           6             view            1   
1     4822           2         purchase            5   
2     8443          24             view            2   
3     1490           9             view            1   
4     3585           8             view            4   

                    timestamp          run_id                          title  \
0  2026-04-02 20:29:11.382578  20260502202911            Calvin Klein CK One   
1  2026-04-02 20:29:11.382633  20260502202911  Eyeshadow Pale

## Step 3: Manual Data Validation Checks

In [3]:
print("="*70)
print("DATA QUALITY VALIDATION CHECKS")
print("="*70)

validation_results = {}

# ============================================================================
# 1. MISSING VALUES CHECK
# ============================================================================

print("\n1. MISSING VALUES CHECK")
print("-" * 70)

missing_data = df_raw.isnull().sum()
missing_percent = (missing_data / len(df_raw)) * 100

missing_check = {
    'total_rows': len(df_raw),
    'columns_with_missing': {},
    'is_passed': True
}

for col in df_raw.columns:
    if missing_data[col] > 0:
        missing_check['columns_with_missing'][col] = {
            'count': int(missing_data[col]),
            'percentage': float(missing_percent[col])
        }
        print(f"  ⚠ {col}: {missing_data[col]} missing ({missing_percent[col]:.2f}%)")
        missing_check['is_passed'] = False
    else:
        print(f"  ✓ {col}: No missing values")

if missing_check['is_passed']:
    print("\n  ✓ PASSED: No missing values detected")
else:
    print(f"\n  ✗ FAILED: {len(missing_check['columns_with_missing'])} columns have missing values")

validation_results['missing_values'] = missing_check

DATA QUALITY VALIDATION CHECKS

1. MISSING VALUES CHECK
----------------------------------------------------------------------
  ✓ user_id: No missing values
  ✓ product_id: No missing values
  ✓ interaction_type: No missing values
  ✓ user_rating: No missing values
  ✓ timestamp: No missing values
  ✓ run_id: No missing values
  ✓ title: No missing values
  ✓ description: No missing values
  ✓ category: No missing values
  ✓ price: No missing values
  ✓ discountPercentage: No missing values
  ✓ product_rating: No missing values
  ✓ stock: No missing values
  ✓ tags: No missing values
  ⚠ brand: 300599 missing (50.10%)
  ✓ sku: No missing values
  ✓ weight: No missing values
  ✓ dimensions: No missing values
  ✓ warrantyInformation: No missing values
  ✓ shippingInformation: No missing values
  ✓ availabilityStatus: No missing values
  ✓ reviews: No missing values
  ✓ returnPolicy: No missing values
  ✓ minimumOrderQuantity: No missing values
  ✓ meta: No missing values
  ✓ images: No 

## Duplicates Check

In [4]:
# ============================================================================
# 2. DUPLICATE ENTRIES CHECK
# ============================================================================

print("\n2. DUPLICATE ENTRIES CHECK")
print("-" * 70)

duplicate_rows = df_raw.duplicated().sum()
duplicate_percent = (duplicate_rows / len(df_raw)) * 100

duplicate_check = {
    'total_duplicates': int(duplicate_rows),
    'duplicate_percentage': float(duplicate_percent),
    'is_passed': duplicate_rows == 0
}

print(f"  Total duplicate rows: {duplicate_rows} ({duplicate_percent:.2f}%)")

# Check for duplicate user-product pairs
user_product_duplicates = df_raw.duplicated(subset=['user_id', 'product_id'], keep=False).sum()
print(f"  Duplicate user-product pairs: {user_product_duplicates}")

if duplicate_check['is_passed']:
    print("\n  ✓ PASSED: No duplicate rows detected")
else:
    print(f"\n  ✗ FAILED: {duplicate_rows} duplicate rows found")

validation_results['duplicates'] = duplicate_check


2. DUPLICATE ENTRIES CHECK
----------------------------------------------------------------------


  Total duplicate rows: 0 (0.00%)
  Duplicate user-product pairs: 518643

  ✓ PASSED: No duplicate rows detected


## Schema & Data Type Validation

In [5]:
# ============================================================================
# 3. SCHEMA & DATA TYPE VALIDATION
# ============================================================================

print("\n3. SCHEMA & DATA TYPE VALIDATION")
print("-" * 70)

expected_schema = {
    'user_id': 'numeric',
    'product_id': 'numeric',
    'interaction_type': 'string',
    'rating': 'numeric',
    'timestamp': 'datetime'
}

schema_check = {
    'expected_columns': list(expected_schema.keys()),
    'actual_columns': df_raw.columns.tolist(),
    'schema_validation': {},
    'is_passed': True
}

# Check columns exist
missing_columns = set(expected_schema.keys()) - set(df_raw.columns)
extra_columns = set(df_raw.columns) - set(expected_schema.keys())

if missing_columns:
    print(f"  ✗ Missing columns: {missing_columns}")
    schema_check['is_passed'] = False

if extra_columns:
    print(f"  ⚠ Extra columns: {extra_columns}")

# Check data types
for col, expected_type in expected_schema.items():
    if col in df_raw.columns:
        actual_type = df_raw[col].dtype
        type_match = False
        
        if expected_type == 'numeric' and actual_type in ['int64', 'float64', 'int32']:
            type_match = True
        elif expected_type == 'string' and actual_type == 'object':
            type_match = True
        elif expected_type == 'datetime':
            type_match = pd.api.types.is_datetime64_any_dtype(df_raw[col])
        
        status = "✓" if type_match else "✗"
        print(f"  {status} {col}: {actual_type} (expected: {expected_type})")
        schema_check['schema_validation'][col] = {
            'expected': expected_type,
            'actual': str(actual_type),
            'is_match': type_match
        }
        if not type_match:
            schema_check['is_passed'] = False

if schema_check['is_passed']:
    print("\n  ✓ PASSED: All columns have correct data types")
else:
    print("\n  ✗ FAILED: Schema validation errors detected")

validation_results['schema'] = schema_check


3. SCHEMA & DATA TYPE VALIDATION
----------------------------------------------------------------------
  ✗ Missing columns: {'rating'}
  ⚠ Extra columns: {'title', 'discountPercentage', 'run_id', 'price', 'tags', 'dimensions', 'reviews', 'category', 'product_rating', 'minimumOrderQuantity', 'returnPolicy', 'description', 'warrantyInformation', 'meta', 'shippingInformation', 'thumbnail', 'brand', 'images', 'stock', 'weight', 'availabilityStatus', 'user_rating', 'sku'}
  ✓ user_id: int64 (expected: numeric)
  ✓ product_id: int64 (expected: numeric)
  ✓ interaction_type: object (expected: string)
  ✗ timestamp: object (expected: datetime)

  ✗ FAILED: Schema validation errors detected


## Range & Format Checks

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
# ============================================================================
# 4. RANGE & FORMAT CHECKS
# ============================================================================

print("\n4. RANGE & FORMAT CHECKS")
print("-" * 70)

range_check = {
    'user_id': {'min': 0, 'max': np.inf},
    'product_id': {'min': 0, 'max': np.inf},
    'rating': {'min': 1, 'max': 5},
    'interaction_type': {'valid_values': ['view', 'click', 'purchase', 'cart']}
}

range_validation = {
    'checks': {},
    'is_passed': True
}

# Check user_id range
user_id_min = df_raw['user_id'].min()
user_id_max = df_raw['user_id'].max()
user_id_valid = (user_id_min >= 0) and (user_id_max > 0)
print(f"  {'✓' if user_id_valid else '✗'} user_id: min={user_id_min}, max={user_id_max} (expected: > 0)")
range_validation['checks']['user_id'] = {'min': int(user_id_min), 'max': int(user_id_max), 'is_valid': user_id_valid}
if not user_id_valid:
    range_validation['is_passed'] = False

# Check product_id range
product_id_min = df_raw['product_id'].min()
product_id_max = df_raw['product_id'].max()
product_id_valid = (product_id_min >= 0) and (product_id_max > 0)
print(f"  {'✓' if product_id_valid else '✗'} product_id: min={product_id_min}, max={product_id_max} (expected: > 0)")
range_validation['checks']['product_id'] = {'min': int(product_id_min), 'max': int(product_id_max), 'is_valid': product_id_valid}
if not product_id_valid:
    range_validation['is_passed'] = False

# Check rating range (1-5)
rating_min = df_raw['rating_x'].min()
rating_max = df_raw['rating_x'].max()
rating_valid = (rating_min >= 1) and (rating_max <= 5)
out_of_range = ((df_raw['rating_x'] < 1) | (df_raw['rating_x'] > 5)).sum()
rating_max = df_raw['rating_x'].max()
rating_valid = (rating_min >= 1) and (rating_max <= 5)
out_of_range = ((df_raw['rating_x'] < 1) | (df_raw['rating_x'] > 5)).sum()
print(f"  {'✓' if rating_valid else '✗'} rating: min={rating_min}, max={rating_max} (expected: 1-5)")
if out_of_range > 0:
    print(f"     ⚠ {out_of_range} out-of-range ratings found")
    range_validation['is_passed'] = False
range_validation['checks']['rating'] = {'min': float(rating_min), 'max': float(rating_max), 'out_of_range': int(out_of_range), 'is_valid': rating_valid}

# Check interaction_type values
valid_interactions = ['view', 'click', 'purchase', 'cart']
unique_interactions = df_raw['interaction_type'].unique()
invalid_interactions = set(unique_interactions) - set(valid_interactions)
interaction_valid = len(invalid_interactions) == 0
print(f"  {'✓' if interaction_valid else '✗'} interaction_type: {unique_interactions} (expected: {valid_interactions})")
if invalid_interactions:
    print(f"     ⚠ Invalid interactions: {invalid_interactions}")
    range_validation['is_passed'] = False
range_validation['checks']['interaction_type'] = {
    'unique_values': unique_interactions.tolist(),
    'valid_values': valid_interactions,
    'is_valid': interaction_valid
}

if range_validation['is_passed']:
    print("\n  ✓ PASSED: All values are within expected ranges")
else:
    print("\n  ✗ FAILED: Out-of-range values detected")

validation_results['range_format'] = range_validation


4. RANGE & FORMAT CHECKS
----------------------------------------------------------------------
  ✓ user_id: min=1, max=10000 (expected: > 0)
  ✓ product_id: min=1, max=30 (expected: > 0)


KeyError: 'rating_x'

## Statistical Summary

In [ ]:
# ============================================================================
# 5. STATISTICAL SUMMARY
# ============================================================================

print("\n5. STATISTICAL SUMMARY")
print("-" * 70)

print("\nNumerical columns:")
print(df_raw[['user_id', 'product_id', 'rating_x', 'rating_y']].describe())

print("\nCategorical columns:")
print(f"\ninteraction_type value counts:")
print(df_raw['interaction_type'].value_counts())

# Distribution info
stats_summary = {
    'total_records': len(df_raw),
    'unique_users': df_raw['user_id'].nunique(),
    'unique_products': df_raw['product_id'].nunique(),
    'interaction_distribution': df_raw['interaction_type'].value_counts().to_dict(),
    'rating_x_distribution': df_raw['rating_x'].value_counts().sort_index().to_dict(),
    'rating_y_distribution': df_raw['rating_y'].value_counts().sort_index().to_dict()
}

print(f"\n\nData Summary:")
print(f"  Total records: {stats_summary['total_records']}")
print(f"  Unique users: {stats_summary['unique_users']}")
print(f"  Unique products: {stats_summary['unique_products']}")
print(f"  Sparsity: {(1 - (stats_summary['total_records'] / (stats_summary['unique_users'] * stats_summary['unique_products']))) * 100:.2f}%")

validation_results['statistical_summary'] = stats_summary

## Generate ydata-profiling HTML Report

In [ ]:
print("\n" + "="*70)
print("GENERATING YDATA-PROFILING REPORT")
print("="*70)

# Create profiling report
print("\nGenerating HTML report (this may take 1-2 minutes)...")

profile = ProfileReport(
    df_raw,
    title='Data Quality & Profiling Report - Raw User Interactions',
    missing_diagrams=None,
    progress_bar=True,
    explorative=True
)

# Save HTML report
report_path = r'e:/dm4ml project/data/data_quality_report.html'
profile.to_file(report_path)

print(f"✓ HTML report generated: {report_path}")
print(f"  File size: {os.path.getsize(report_path) / (1024*1024):.2f} MB")
print(f"\n  Open in browser: file:///{os.path.abspath(report_path)}")

## Save Validation Report

In [ ]:
# ============================================================================
# SAVE VALIDATION RESULTS
# ============================================================================

print("\n" + "="*70)
print("DATA QUALITY VALIDATION REPORT")
print("="*70)

# Create comprehensive report
final_report = {
    'generated_at': datetime.now().isoformat(),
    'data_source': 'user_interactions.csv',
    'validation_checks': validation_results,
    'overall_status': all([
        validation_results['missing_values']['is_passed'],
        validation_results['duplicates']['is_passed'],
        validation_results['schema']['is_passed'],
        validation_results['range_format']['is_passed']
    ])
}


# Print summary
print(f"\n{'='*70}")
print("VALIDATION SUMMARY")
print(f"{'='*70}")

checks = [
    ('Missing Values', validation_results['missing_values']['is_passed']),
    ('Duplicate Entries', validation_results['duplicates']['is_passed']),
    ('Schema Validation', validation_results['schema']['is_passed']),
    ('Range & Format', validation_results['range_format']['is_passed'])
]

for check_name, status in checks:
    status_str = "✓ PASSED" if status else "✗ FAILED"
    print(f"{check_name:.<30} {status_str}")

print(f"\n{'OVERALL STATUS':.<30} {'✓ ALL CHECKS PASSED' if final_report['overall_status'] else '✗ SOME CHECKS FAILED'}")

print(f"\n Generated Reports:")
print(f"   HTML Report: data_quality_report.html")